In [1]:
library(fixest)
library(modelsummary)
library(gt)
library(marginaleffects)
library(ggeffects)
library(ggplot2)
library(dplyr)
library(sjPlot)
library(scales)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Learn more about sjPlot with 'browseVignettes("sjPlot")'.


Attaching package: ‘scales’


The following object is masked from ‘package:fixest’:

    pvalue




In [2]:
df_path='../../data/dropbox/LaborDivision/data/deriveddata/intl_noalpha/regression_table.csv'
data=read.csv(df_path)

In [3]:
head(data)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,29768070,65683243,JPN,F,2008,51,F,2008,5,Advanced,invalid
2,31240446,65683243,JPN,F,2008,51,F,2008,5,Advanced,invalid
3,29768070,79129017,JPN,M,2008,51,M,2000,98,Advanced,invalid
4,31240446,79129017,JPN,M,2008,51,M,2000,98,Advanced,invalid
5,39595623,79129017,JPN,F,2013,18,M,2000,98,Advanced,1
6,40710917,79129017,JPN,F,2013,51,M,2000,98,Advanced,1


In [4]:
# Step 1: Count the number of unique groups each paper belongs to
group_counts <- data %>%
  group_by(ost_bk) %>%
  summarise(group_count = n_distinct(ST))

In [5]:

# Step 2: Filter out papers that only belong to one group
papers_with_multiple_groups <- group_counts %>%
  filter(group_count > 1) %>%
  pull(ost_bk)

# Step 3: Filter the original DataFrame to keep only the papers with multiple groups
filtered_df <- data %>%
  filter(ost_bk %in% papers_with_multiple_groups)

# Now, filtered_df contains only papers that belong to more than one group

In [6]:
data = filtered_df

In [7]:
head(data)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,29768070,65683243,JPN,F,2008,51,F,2008,5,Advanced,invalid
2,31240446,65683243,JPN,F,2008,51,F,2008,5,Advanced,invalid
3,29768070,79129017,JPN,M,2008,51,M,2000,98,Advanced,invalid
4,31240446,79129017,JPN,M,2008,51,M,2000,98,Advanced,invalid
5,43269102,79129017,JPN,M,2015,51,M,2000,98,Advanced,1
6,46664474,79129017,JPN,M,2016,18,M,2000,98,Advanced,1


In [8]:
data$npub = log(data$npub)
data$funded = factor(data$funded, levels=c(0,1), labels=c('NotFund','Funded'))
data$fdep = ifelse(data$role=='F', 1, 0)
data$ldep = ifelse(data$role=='L', 1, 0)
data$mdep = ifelse(data$role=='M', 1, 0)

In [9]:
length(unique(data$ost_bk))

[1] 1850447

In [10]:
fund_data = data[!is.na(data$funded),]

In [11]:
length(unique(fund_data$ost_bk))

[1] 703987

## model with fixed effect with fixest package

In [12]:
model1 <- feglm(fdep~gender+npub+ST|ost_bk,data = data,family = binomial())

In [13]:
summary(model1)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 12,164,877 
Fixed-effects: ost_bk: 1,850,447
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error   t value  Pr(>|t|)    
genderM      -0.246825   0.002546  -96.9618 < 2.2e-16 ***
genderU      -0.282034   0.003497  -80.6604 < 2.2e-16 ***
npub         -0.196587   0.000585 -335.9455 < 2.2e-16 ***
STDeveloping  0.556199   0.004731  117.5668 < 2.2e-16 ***
STLagging     0.251358   0.004413   56.9637 < 2.2e-16 ***
STProficient  0.627145   0.002365  265.2168 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -4,740,149.2   Adj. Pseudo R2: -0.270698
           BIC: 39,668,705.9     Squared Cor.:  0.080201

In [14]:
model2 <- feglm(fdep~gender+npub+ST+funded|ost_bk,data = fund_data,family = binomial())
summary(model2)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 4,640,948 
Fixed-effects: ost_bk: 703,987
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error   t value  Pr(>|t|)    
genderM      -0.212306   0.004213  -50.3914 < 2.2e-16 ***
genderU      -0.253528   0.005764  -43.9870 < 2.2e-16 ***
npub         -0.201073   0.000937 -214.6026 < 2.2e-16 ***
STDeveloping  0.839123   0.010054   83.4585 < 2.2e-16 ***
STLagging     0.437702   0.008909   49.1283 < 2.2e-16 ***
STProficient  0.706974   0.004376  161.5452 < 2.2e-16 ***
fundedFunded  1.155209   0.004819  239.7048 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -1,768,982.9   Adj. Pseudo R2: -0.251933
           BIC: 14,344,575.8     Squared Cor.:  0.094555

In [15]:
model3 <- feglm(fdep~gender+npub+ST+funded+ST*funded|ost_bk,data = fund_data,family = binomial())
summary(model3)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 4,640,948 
Fixed-effects: ost_bk: 703,987
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error    t value   Pr(>|t|)    
genderM                   -0.212548   0.004216  -50.41816  < 2.2e-16 ***
genderU                   -0.253520   0.005767  -43.96342  < 2.2e-16 ***
npub                      -0.200939   0.000938 -214.13954  < 2.2e-16 ***
STDeveloping               0.724369   0.013923   52.02817  < 2.2e-16 ***
STLagging                  0.397391   0.012987   30.59798  < 2.2e-16 ***
STProficient               0.411912   0.010376   39.69982  < 2.2e-16 ***
fundedFunded               1.002782   0.007984  125.59595  < 2.2e-16 ***
STDeveloping:fundedFunded  0.149156   0.021509    6.93462 4.0732e-12 ***
STLagging:fundedFunded    -0.027599   0.019271   -1.43211 1.5211e-01    
STProficient:fundedFunded  0.424949   0.013505   31.46700  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 

In [16]:
model4 <- feglm(mdep~gender+npub+ST|ost_bk,data = data,family = binomial())
model5 <- feglm(mdep~gender+npub+ST+funded|ost_bk,data = fund_data,family = binomial())
model6 <- feglm(mdep~gender+npub+ST+funded+ST*funded|ost_bk,data = fund_data,family = binomial())

NOTE: 64,675 fixed-effects (129,350 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 18,635 fixed-effects (37,270 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 18,635 fixed-effects (37,270 observations) removed because of only 0 (or only 1) outcomes.



In [17]:
summary(model4)

GLM estimation, family = binomial, Dep. Var.: mdep
Observations: 12,035,527 
Fixed-effects: ost_bk: 1,785,772
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error    t value   Pr(>|t|)    
genderM       0.022367   0.002050   10.91071  < 2.2e-16 ***
genderU       0.020754   0.002710    7.65772 1.8926e-14 ***
npub         -0.129352   0.000450 -287.44046  < 2.2e-16 ***
STDeveloping  0.092192   0.004220   21.84738  < 2.2e-16 ***
STLagging     0.164269   0.003921   41.89560  < 2.2e-16 ***
STProficient -0.190100   0.002111  -90.03503  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -6,607,288.7   Adj. Pseudo R2: -0.146808
           BIC: 42,328,783.1     Squared Cor.:  0.113596

In [18]:
summary(model5)

GLM estimation, family = binomial, Dep. Var.: mdep
Observations: 4,603,678 
Fixed-effects: ost_bk: 685,352
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error    t value   Pr(>|t|)    
genderM      -0.003823   0.003338   -1.14541 2.5204e-01    
genderU       0.009350   0.004396    2.12690 3.3428e-02 *  
npub         -0.136641   0.000705 -193.68362  < 2.2e-16 ***
STDeveloping  0.054994   0.007982    6.88985 5.5852e-12 ***
STLagging     0.151948   0.007263   20.91954  < 2.2e-16 ***
STProficient -0.101333   0.003557  -28.48775  < 2.2e-16 ***
fundedFunded -0.681091   0.003614 -188.48470  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -2,522,085.0   Adj. Pseudo R2: -0.144144
           BIC: 15,559,198.7     Squared Cor.:  0.116715

In [19]:
summary(model6)

GLM estimation, family = binomial, Dep. Var.: mdep
Observations: 4,603,678 
Fixed-effects: ost_bk: 685,352
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error    t value   Pr(>|t|)    
genderM                   -0.003589   0.003339   -1.07471 2.8250e-01    
genderU                    0.009213   0.004398    2.09496 3.6175e-02 *  
npub                      -0.137229   0.000707 -194.13502  < 2.2e-16 ***
STDeveloping               0.090241   0.011387    7.92484 2.2844e-15 ***
STLagging                  0.161543   0.010159   15.90106  < 2.2e-16 ***
STProficient               0.140206   0.008087   17.33735  < 2.2e-16 ***
fundedFunded              -0.578069   0.005774 -100.10977  < 2.2e-16 ***
STDeveloping:fundedFunded -0.030767   0.016863   -1.82452 6.8073e-02 .  
STLagging:fundedFunded     0.035702   0.015590    2.29012 2.2014e-02 *  
STProficient:fundedFunded -0.345687   0.010717  -32.25565  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 

In [20]:
model7 <- feglm(ldep~gender+npub+ST|ost_bk,data = data,family = binomial())
summary(model7)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 12,164,877 
Fixed-effects: ost_bk: 1,850,447
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error   t value  Pr(>|t|)    
genderM       0.234850   0.002885   81.4149 < 2.2e-16 ***
genderU       0.210898   0.003782   55.7629 < 2.2e-16 ***
npub          0.443330   0.000784  565.3365 < 2.2e-16 ***
STDeveloping -0.638038   0.005843 -109.2034 < 2.2e-16 ***
STLagging    -0.448658   0.005306  -84.5580 < 2.2e-16 ***
STProficient -0.278966   0.002748 -101.5086 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -4,509,892.3   Adj. Pseudo R2: -0.226304
           BIC: 39,208,192.3     Squared Cor.:  0.117984

In [21]:
model8 <- feglm(ldep~gender+npub+ST+funded|ost_bk,data = fund_data,family = binomial())
summary(model8)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 4,640,948 
Fixed-effects: ost_bk: 703,987
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error  t value  Pr(>|t|)    
genderM       0.243926   0.004722  51.6607 < 2.2e-16 ***
genderU       0.194893   0.006155  31.6643 < 2.2e-16 ***
npub          0.454458   0.001255 362.0087 < 2.2e-16 ***
STDeveloping -0.644311   0.010724 -60.0827 < 2.2e-16 ***
STLagging    -0.503128   0.009631 -52.2404 < 2.2e-16 ***
STProficient -0.354828   0.004554 -77.9139 < 2.2e-16 ***
fundedFunded  0.170110   0.004660  36.5025 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -1,712,763.6   Adj. Pseudo R2: -0.223472
           BIC: 14,232,137.3     Squared Cor.:  0.121092

In [22]:
model9 <- feglm(ldep~gender+npub+ST+funded+ST*funded|ost_bk,data = fund_data,family = binomial())
summary(model9)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 4,640,948 
Fixed-effects: ost_bk: 703,987
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error   t value   Pr(>|t|)    
genderM                    0.242675   0.004723  51.38455  < 2.2e-16 ***
genderU                    0.193756   0.006156  31.47452  < 2.2e-16 ***
npub                       0.455083   0.001257 361.94078  < 2.2e-16 ***
STDeveloping              -0.483272   0.015582 -31.01558  < 2.2e-16 ***
STLagging                 -0.331833   0.013278 -24.99186  < 2.2e-16 ***
STProficient              -0.334394   0.010476 -31.92013  < 2.2e-16 ***
fundedFunded               0.245313   0.007245  33.85991  < 2.2e-16 ***
STDeveloping:fundedFunded -0.326029   0.022407 -14.55055  < 2.2e-16 ***
STLagging:fundedFunded    -0.341146   0.020426 -16.70154  < 2.2e-16 ***
STProficient:fundedFunded -0.055540   0.013779  -4.03088 5.5569e-05 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' '

In [34]:
models = list("(1)"=model1,"(2)"=model2,"(3)"=model3,"(1)"=model4,"(2)"=model5,"(3)"=model6,"(1)"=model7,"(2)"=model8,"(3)"=model9)
table = modelsummary(models, output='gt', stars=TRUE,
            coef_map=c('genderM'='Male','genderU'='Gender(Unknown)','npub'='Log no.pub',
                          'STProficient'='Proficient','STDeveloping'='Developing','STLagging'='Lagging',
                          'fundedFunded'='Funded',
                          'STProficient:fundedFunded'='Proficient × Funded',
                          'STDeveloping:fundedFunded'='Developing × Funded',
                          'STLagging:fundedFunded'='Lagging × Funded'),
            gof_map = c("nobs", "r.squared","adj.r.squared"))
table |> 
tab_spanner(label='First Author',columns=c(2:4)) |> 
tab_spanner(label='Middle Author',columns=c(5:7))|>
tab_spanner(label='Last Author',columns=c(8:10))|>
gtsave(file='regression.html')

In [24]:
options(repr.plot.width=12, repr.plot.height=4)

In [25]:
color_scheme_path='../../data/dropbox/LaborDivision/data/additionaldata/cntry_classification/color_scheme.csv'
color_scheme=read.csv(color_scheme_path)

In [26]:
color_scheme = color_scheme[color_scheme$type == "st", ]

In [27]:
color_palette <- setNames(color_scheme$color, color_scheme$class)
color_palette

Advanced Proficient Developing    Lagging 
 "#FFB100"  "#35b779"  "#5FB6D9"  "#936994"

In [28]:
unique(fund_data$funded)

[1] Funded  NotFund
Levels: NotFund Funded

In [29]:
# Define a function that combines both steps
get_pred_data_ordered <- function(model) {
  # Step 1: Generate prediction data from the model without plotting
  pred_data <- plot_model(
    model, 
    type = 'pred', 
    terms = c("funded [NotFund, Funded]", "ST [Advanced,Proficient,Developing,Lagging]"),
    plot = FALSE  # This prevents plotting and returns the data
  )
  
  # Step 2: Add 'order' column to the prediction data
  pred_data_ordered <- pred_data$data %>%
    mutate(
      order = case_when(
        x == 1 & group == "Lagging"   ~ 2,
        x == 1 & group == "Developing" ~ 2.05,
        x == 1 & group == "Proficient" ~ 2.1,
        x == 1 & group == "Advanced"    ~ 2.15,
        x == 2 & group == "Lagging"   ~ 1,
        x == 2 & group == "Developing" ~ 1.05,
        x == 2 & group == "Proficient" ~ 1.1,
        x == 2 & group == "Advanced"    ~ 1.15
      )
    )
  
  # Return the ordered prediction data
  return(pred_data_ordered)
}

In [30]:
# Define a function to create the plot with a dynamic x-axis label
create_plot <- function(data, x_axis_label) {
  ggplot(data, aes(x = predicted, y = order, color = group, group=group)) +
    geom_point(size = 3.5) +
    geom_errorbarh(aes(xmin = conf.low, xmax = conf.high), height = 0.05) + # Error bars
    geom_line(linetype = "dashed", linewidth=0.2)+
    scale_y_continuous(
      breaks = c(1.05, 2.05),  # Specify the breaks for Funded and Not Funded
      labels = c('W/Funding','W/O Funding')
    ) +
    scale_x_continuous(
      name = x_axis_label,  # Use the x_axis_label parameter for the x-axis label
      labels = label_percent(accuracy = 1)  # Convert x-axis values to percentages
    ) +
    scale_color_manual(values = color_palette, name = "Scientific Capacity") +
    theme_minimal() +
    theme(
      axis.text.y = element_text(size = 10),
      axis.text.x = element_text(size = 10),
      axis.title.y = element_blank(),  # No y-axis title needed
      axis.title.x = element_text(size = 10),
      legend.title = element_text(size = 10),
      legend.text = element_text(size = 8),
      panel.grid.major = element_blank(),  # Remove horizontal grid lines for clarity
      panel.grid.minor = element_blank(),
      axis.line = element_line(linewidth = 0.5, linetype = "solid", colour = "black"), 
    )
}

In [31]:
pred_data_ordered = get_pred_data_ordered(model3)
pdf("MEF_First_plot.pdf", family = "Helvetica")
plot = create_plot(pred_data_ordered,'Probability of being first author')
ggsave("MEF_First_plot.pdf", plot, width = 5, height = 2)

In [32]:
pred_data_ordered = get_pred_data_ordered(model6)
pdf("MEF_Middle_plot.pdf", family = "Helvetica")
plot = create_plot(pred_data_ordered,'Probability of being middle author')
ggsave("MEF_Middle_plot.pdf", plot, width = 5, height = 2)

In [33]:
pred_data_ordered = get_pred_data_ordered(model9)
pdf("MEF_Last_plot.pdf", family = "Helvetica")
plot = create_plot(pred_data_ordered,'Probability of being last author')
ggsave("MEF_Last_plot.pdf", plot, width = 5, height = 2)